In [ ]:
from datetime import date
import json
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'scripts' / '79_dndt_dndf_preflight.py').is_file()
)
CONFIG = PROJECT_ROOT / 'configs' / 'dndt_dndf_two_day.json'
RUN_ID = f"dndt-dndf-{date.today():%Y%m%d}"
PYTHON = sys.executable

def run(command):
    print(' '.join(map(str, command)), flush=True)
    return subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print({'project_root': str(PROJECT_ROOT), 'config': str(CONFIG), 'run_id': RUN_ID})


In [ ]:
run([PYTHON, 'scripts/79_dndt_dndf_preflight.py', '--config', str(CONFIG), '--device', 'cuda'])


In [ ]:
run([PYTHON, 'scripts/80_run_dndt_dndf_track_a.py', '--config', str(CONFIG), '--run-id', RUN_ID, '--resume'])


In [ ]:
run([PYTHON, 'scripts/81_run_dndt_dndf_track_b.py', '--config', str(CONFIG), '--run-id', RUN_ID, '--stage', 'candidates', '--resume'])
run([PYTHON, 'scripts/81_run_dndt_dndf_track_b.py', '--config', str(CONFIG), '--run-id', RUN_ID, '--stage', 'final', '--resume'])
run([PYTHON, 'scripts/81_run_dndt_dndf_track_b.py', '--config', str(CONFIG), '--run-id', RUN_ID, '--stage', 'fusion', '--resume'])
run([PYTHON, 'scripts/81_run_dndt_dndf_track_b.py', '--config', str(CONFIG), '--run-id', RUN_ID, '--stage', 'prespecified_ladder_v2', '--resume'])
run([PYTHON, 'scripts/81_run_dndt_dndf_track_b.py', '--config', str(CONFIG), '--run-id', RUN_ID, '--stage', 'shuffle', '--resume'])


In [ ]:
run([PYTHON, 'scripts/82_make_dndt_dndf_evidence.py', '--config', str(CONFIG), '--run-id', RUN_ID])


In [ ]:
import pandas as pd

configuration = json.loads(CONFIG.read_text(encoding='utf-8'))
run_dir = Path(configuration['run_root']) / RUN_ID
receipt_paths = sorted(run_dir.rglob('completion.json'))
manifests = [
    run_dir / 'track_a_manifest.json',
    run_dir / 'track_b' / 'candidates_manifest.json',
    run_dir / 'track_b' / 'final_manifest.json',
    run_dir / 'track_b' / 'fusion_manifest.json',
    run_dir / 'track_b' / 'prespecified_ladder_v2_manifest.json',
    run_dir / 'track_b' / 'shuffle_manifest.json',
]
progress = [json.loads(path.read_text(encoding='utf-8')) for path in manifests if path.is_file()]
completed = sum(int(item.get('completed_units', 0)) for item in progress)
total = sum(int(item.get('total_units', 0)) for item in progress)
pending = max(total - completed, 0)
print({'completed': completed, 'pending': pending, 'receipts': len(receipt_paths)})
summary = run_dir / 'evidence' / 'final_summary.csv'
display(pd.read_csv(summary) if summary.is_file() else pd.DataFrame())
